# 06 — Final Test Inference (Phases 22-24)

Runs the frozen pipeline (model + IDF tables + vectorizers + threshold from
notebook 04/05) over the **full TEST set** and writes the two required
submission files, then runs the official validator.

Equivalent one-liner (what actually gets used at submission time):
```bash
python -m src.inference --split test --n-jobs-normalize <vCPUs>
python -m src.validation
```


In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import joblib
from src import config
from src.inference import run_pipeline
from src.model import SklearnModelWrapper, RuleBasedModel


In [ ]:
idf_tables = joblib.load(config.MODELS_DIR / "idf_tables.joblib")
threshold_info = json.loads((config.MODELS_DIR / "threshold.json").read_text())
model_path = config.MODELS_DIR / "final_model.joblib"
model_wrapper = SklearnModelWrapper.load(model_path) if model_path.exists() else RuleBasedModel()
threshold_info


In [ ]:
result = run_pipeline(
    split="test",
    model_wrapper=model_wrapper,
    name_idf=idf_tables["name_idf"],
    addr_idf=idf_tables["addr_idf"],
    threshold=threshold_info["threshold"],
    output_dir=config.OUTPUT_DIR,
    chunk_size=config.S1_CHUNK_SIZE,
    n_jobs_normalize=8,   # set to the instance's vCPU count
    write_outputs=True,
)
result


## Run the official validator

In [ ]:
from src.validation import run_official_validator
rc = run_official_validator()
print("validator exit code:", rc, "(0 = PASS)")


If this prints `PASS`, `output/matching_results.tsv` and
`output/candidate_pairs.tsv` are ready to submit. Also sanity-check the
summary numbers requested in the final deliverables (candidate count,
predicted match count, threshold used, validation F0.5 from notebook 04/05)
before packaging the submission zip.